## Load packages

In [ ]:
suppressMessages(library(dplyr))
suppressMessages(library(openxlsx))
suppressMessages(library(flowCore))

In [ ]:
base_dir <- "~/CODEX_08262025"  ## Base dir to input data

## Load Meta data

In [ ]:
meta1 <- read.xlsx(paste0(base_dir,"/PD-TMA-Reg Deidentified_mod.xlsx")) %>%
    dplyr::filter(condition != "N")

meta2 <- read.xlsx(paste0(base_dir,"/TMAmeta.xlsx"), sheet = "afterExclude") %>% 
    dplyr::filter(Use == "Y")
table(meta2$Code)

In [ ]:
keep_codes <- names(table(meta2$Code))[table(meta2$Code) > 1]
keep_codes

meta2 <- meta2[meta2$Code %in% keep_codes, ]
meta2

In [ ]:
meta <- merge(meta1, meta2, by = c('Code'))
dim(meta)

## Load Channel names
### TMA1

In [ ]:
channels <- read.table(paste0(base_dir, "/DATA/TMA1/channelnames.txt"), sep = "\t", header = F)$V1
channels

fcs_files_1 <- list.files(full.names = T, path = paste0(base_dir, "/DATA/TMA1/fcs/compensated"),
                          pattern = "fcs$")

## Load FCS files with transformation=T

In [ ]:
celldata1 <- do.call(rbind, 
                     lapply(fcs_files_1, 
                            function(x) {
                                t_dt <- read.FCS(x, 
                                                 transformation = T, 
                                                 truncate_max_range = F)@exprs %>% 
                                data.frame
                                
                                colnames(t_dt)[which(grepl("^cyc", colnames(t_dt)))] <- channels
                                t_dt$TMA <- 1
                                t_dt
                            }))

## Load Channel names
### TMA2

In [ ]:
channels <- read.table(paste0(base_dir, "/DATA/TMA2/channelnames.txt"), sep = "\t", header = F)$V1
channels

fcs_files_2 <- list.files(full.names = T, path = paste0(base_dir, "/DATA/TMA2/fcs/compensated"),
                          pattern = "fcs$")

## Load FCS files with transformation=T

In [ ]:
celldata2 <- do.call(rbind, 
                     lapply(fcs_files_2, 
                            function(x) {
                                t_dt <- read.FCS(x, 
                                                 transformation = T, 
                                                 truncate_max_range = F)@exprs %>% 
                                data.frame
                                
                                colnames(t_dt)[which(grepl("^cyc", colnames(t_dt)))] <- channels
                                t_dt$TMA <- 2
                                t_dt
                            }))

## Exclude non protein markers

In [ ]:
celldata <- rbind(celldata1, celldata2)
colnames(celldata)[2] <- "Region"

celldata <- celldata[, -which(grepl("Blank|^Hoechst|^Ch[1-9]", colnames(celldata)))]

In [ ]:
colnames(celldata)
colnames(meta)

In [ ]:
merged <- merge(celldata, meta, by = c("TMA", "Region"))

## Write out final cell data to an Excel file

In [ ]:
write.xlsx(merged, paste0(base_dir, "/celldata_wLinearize.xlsx"))

In [ ]:
table(paste0(merged$TMA, "_", merged$Region), merged$Code)